<a href="https://colab.research.google.com/github/Judiipiie/Cats_vs_Dog/blob/main/cats_vs_dogs_colab_final_Progress_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Section 1 — Setup

In [ ]:
!pip install grad-cam -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 76.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os, random, shutil, zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


---
## Section 2 — Upload & Prepare Dataset
Run the cell below — a **Choose Files** button will appear. Click it and select your `train.zip`.

In [ ]:
from google.colab import files
print('Click Choose Files and select train.zip')
uploaded = files.upload()

Click Choose Files and select train.zip


Saving train.zip to train.zip


In [ ]:
zip_path = None
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        zip_path = f'/content/{fname}'
        break
print(f'Found zip: {zip_path}')


print('Unzipping...')
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/raw_data')
print('Done!')


raw_contents = os.listdir('/content/raw_data')
print('Contents:', raw_contents)

Found zip: /content/train.zip
Unzipping...
Done!
Contents: ['train']


In [ ]:
# Set LOCAL_TRAIN to wherever the images are
# If unzip created a 'train' subfolder:
if os.path.exists('/content/raw_data/train'):
    LOCAL_TRAIN = '/content/raw_data/train'
else:
    LOCAL_TRAIN = '/content/raw_data'
print(f'Using: {LOCAL_TRAIN}')
print(f'Sample files: {os.listdir(LOCAL_TRAIN)[:5]}')

# Create split folders
for split in ['train', 'val', 'test']:
    for cls in ['cat', 'dog']:
        os.makedirs(f'data/{split}/{cls}', exist_ok=True)

# Collect images
all_cats = sorted([f for f in os.listdir(LOCAL_TRAIN) if f.startswith('cat') and f.endswith('.jpg')])
all_dogs = sorted([f for f in os.listdir(LOCAL_TRAIN) if f.startswith('dog') and f.endswith('.jpg')])
random.shuffle(all_cats)
random.shuffle(all_dogs)
print(f'Found {len(all_cats)} cats, {len(all_dogs)} dogs')

# Split 80/10/10
def split_and_copy(files_list, cls_name):
    n = len(files_list)
    train_end = int(0.8 * n)
    val_end   = int(0.9 * n)
    splits = {'train': files_list[:train_end],
              'val':   files_list[train_end:val_end],
              'test':  files_list[val_end:]}
    for split, flist in splits.items():
        for fname in flist:
            shutil.copy(f'{LOCAL_TRAIN}/{fname}', f'data/{split}/{cls_name}/{fname}')

print('Splitting dataset...')
split_and_copy(all_cats, 'cat')
split_and_copy(all_dogs, 'dog')

for split in ['train', 'val', 'test']:
    cats = len(os.listdir(f'data/{split}/cat'))
    dogs = len(os.listdir(f'data/{split}/dog'))
    print(f'{split}: {cats} cats, {dogs} dogs')

Using: /content/raw_data/train
Sample files: ['cat.6928.jpg', 'dog.6826.jpg', 'dog.665.jpg', 'dog.5673.jpg', 'cat.7931.jpg']
Found 12500 cats, 12500 dogs
Splitting dataset...
train: 10000 cats, 10000 dogs
val: 1250 cats, 1250 dogs
test: 1250 cats, 1250 dogs


In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder('data/train', transform=train_transform)
val_ds   = datasets.ImageFolder('data/val',   transform=val_test_transform)
test_ds  = datasets.ImageFolder('data/test',  transform=val_test_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

CLASS_NAMES = train_ds.classes
print('Classes:', CLASS_NAMES)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

Classes: ['cat', 'dog']
Train: 20000 | Val: 2500 | Test: 2500


---
## Section 3 — Training Helpers

In [ ]:
def train_model(model, train_loader, val_loader, epochs=15, lr=1e-3):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
        train_loss = running_loss / total
        train_acc  = correct / total

        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                running_loss += loss.item() * imgs.size(0)
                _, preds = outputs.max(1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)
        val_loss = running_loss / total
        val_acc  = correct / total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        scheduler.step()
        print(f'Epoch {epoch+1:02d}/{epochs} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    return model, history


def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            _, preds = outputs.max(1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'],   label='Val')
    axes[0].set_title(f'{title} — Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'],   label='Val')
    axes[1].set_title(f'{title} — Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_")}_history.png', dpi=100)
    plt.show()


def plot_confusion(labels, preds, title):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{title} — Confusion Matrix')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_")}_confusion.png', dpi=100)
    plt.show()
    print(classification_report(labels, preds, target_names=CLASS_NAMES))

---
## Section 4 — ResNet50

In [ ]:
resnet = models.resnet50(weights='IMAGENET1K_V1')

for param in resnet.parameters():
    param.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, 2)

print('Phase 1: training head only (5 epochs)')
resnet, history_rn_p1 = train_model(resnet, train_loader, val_loader, epochs=5, lr=1e-3)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 145MB/s]


Phase 1: training head only (5 epochs)


In [ ]:
for param in resnet.layer4.parameters():
    param.requires_grad = True
for param in resnet.fc.parameters():
    param.requires_grad = True

print('Phase 2: fine-tuning layer4 + head (10 epochs)')
resnet, history_rn_p2 = train_model(resnet, train_loader, val_loader, epochs=10, lr=1e-4)

history_resnet = {k: history_rn_p1[k] + history_rn_p2[k] for k in history_rn_p1}
plot_history(history_resnet, 'ResNet50')

In [ ]:
labels_rn, preds_rn = evaluate_model(resnet, test_loader)
plot_confusion(labels_rn, preds_rn, 'ResNet50')
torch.save(resnet.state_dict(), 'resnet50_cats_dogs.pth')
print('ResNet50 saved.')

NameError: name 'evaluate_model' is not defined

---
## Section 5 — MobileNetV2

In [ ]:
mobilenet = models.mobilenet_v2(weights='IMAGENET1K_V1')

for param in mobilenet.parameters():
    param.requires_grad = False
mobilenet.classifier[1] = nn.Linear(mobilenet.last_channel, 2)

print('Phase 1: training head only (5 epochs)')
mobilenet, history_mn_p1 = train_model(mobilenet, train_loader, val_loader, epochs=5, lr=1e-3)

In [ ]:
for param in mobilenet.features[-3:].parameters():
    param.requires_grad = True

print('Phase 2: fine-tuning last 3 blocks (10 epochs)')
mobilenet, history_mn_p2 = train_model(mobilenet, train_loader, val_loader, epochs=10, lr=1e-4)

history_mobilenet = {k: history_mn_p1[k] + history_mn_p2[k] for k in history_mn_p1}
plot_history(history_mobilenet, 'MobileNetV2')

In [ ]:
labels_mn, preds_mn = evaluate_model(mobilenet, test_loader)
plot_confusion(labels_mn, preds_mn, 'MobileNetV2')
torch.save(mobilenet.state_dict(), 'mobilenetv2_cats_dogs.pth')
print('MobileNetV2 saved.')